This will give you data on what the total numbers of data points per classification, per observation.

In [ ]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)
import numpy as np

allData = pd.read_csv("allData.csv")

# Just going to define the classes here
allDataClasses = ["Tree cover", "Grassland", "Shrubland",
                  "Bare/sparse vegetation", "Permanent water bodies",
                  "Built-up", "Cropland", "Herbaceous Wetland"]

# Going to have these align with class type
studentClassCounts = []
modelClassCounts = []

# Counting everything up
for _ in allDataClasses:
    student_count = 0
    model_count = 0
    for i in allData['crosswalked label (student data)']:
        if i == _:
          student_count += 1
    for i in allData['Model label (worldcover)']:
        if i == _:
          model_count += 1
    studentClassCounts.append(student_count)
    modelClassCounts.append(model_count)

# Organize into a single dataframe
classNumbers = pd.DataFrame({
    'Class Name': allDataClasses,
    'Ground Instances': studentClassCounts,
    'Model Instances': modelClassCounts
})

# Getting a row for total values
new_row = pd.DataFrame([{'Class Name': 'Total',
                         'Ground Instances': len(allData['crosswalked label (student data)']),
                         'Model Instances': len(allData['Model label (worldcover)'])}])

# Adding in a new row for column totals
classNumbers = pd.concat([classNumbers, new_row], ignore_index=True)

# Checking data validity
EXPECTED_ROWS = 3600

if len(allData) != EXPECTED_ROWS:
    raise ValueError(f"Expected {EXPECTED_ROWS} rows, found {len(allData)}")

for column in [
    "crosswalked label (student data)",
    "Model label (worldcover)"
]:
    invalid = ~allData[column].isin(allDataClasses)

    if invalid.any():
        invalid_values = allData.loc[invalid, column].value_counts(dropna=False)
        raise ValueError(
            f"Unrecognized labels in '{column}':\n{invalid_values}"
        )

classNumbers.style.hide(axis='index')

Class Name,Ground Instances,Model Instances
Tree cover,621,678
Grassland,529,162
Shrubland,105,0
Bare/sparse vegetation,15,53
Permanent water bodies,233,228
Built-up,2097,2451
Cropland,0,28
Herbaceous Wetland,0,0
Total,3600,3600


This will give you the corresponding Confusion Matrix for all of the data

In [ ]:
# Get the true labels and predicted labels
true_labels = allData['crosswalked label (student data)']
predicted_labels = allData['Model label (worldcover)']

# Generate the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels, labels=allDataClasses)

# Convert the confusion matrix to a DataFrame for better visualization
cm_df = pd.DataFrame(cm, index=allDataClasses, columns=allDataClasses)

print("Confusion Matrix:")
display(cm_df)

Confusion Matrix:


,Tree cover,Grassland,Shrubland,Bare/sparse vegetation,Permanent water bodies,Built-up,Cropland,Herbaceous Wetland
Tree cover,273,12,0,11,8,315,2,0
Grassland,83,112,0,10,0,301,23,0
Shrubland,17,3,0,1,3,79,2,0
Bare/sparse vegetation,0,6,0,0,0,9,0,0
Permanent water bodies,11,0,0,2,217,3,0,0
Built-up,294,29,0,29,0,1744,1,0
Cropland,0,0,0,0,0,0,0,0
Herbaceous Wetland,0,0,0,0,0,0,0,0


These are associate metrics with the Confusion Matrix

In [ ]:
# User's Accuracy (Precision)
user_accuracy = precision_score(
    true_labels,
    predicted_labels,
    labels=allDataClasses,
    average=None,
    zero_division=np.nan
)

# Producer's Accuracy (Recall)
producer_accuracy = recall_score(
    true_labels,
    predicted_labels,
    labels=allDataClasses,
    average=None,
    zero_division=np.nan
)

# F1 Score
f1 = f1_score(
    true_labels,
    predicted_labels,
    labels=allDataClasses,
    average=None,
    zero_division=np.nan
)

contextual_arr = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=allDataClasses
).sum(axis=1)

# Overall Accuracy
overall_accuracy = accuracy_score(
    true_labels,
    predicted_labels
)

print("Overall Accuracy:", overall_accuracy)

netMetrics = pd.DataFrame({
      "Class Name" : allDataClasses,
      "User Accuracy" : user_accuracy,
      "Producer Accuracy" : producer_accuracy,
      "F1 Score" : f1,
      "Ground-Truth Count" : contextual_arr
})

netMetrics.style.hide(axis='index')

Overall Accuracy: 0.6516666666666666


Class Name,User Accuracy,Producer Accuracy,F1 Score,Ground-Truth Count
Tree cover,0.402655,0.439614,0.420323,621
Grassland,0.691358,0.211720,0.324168,529
Shrubland,nan,0.000000,0.000000,105
Bare/sparse vegetation,0.000000,0.000000,0.000000,15
Permanent water bodies,0.951754,0.931330,0.941432,233
Built-up,0.711546,0.831664,0.766931,2097
Cropland,0.000000,nan,0.000000,0
Herbaceous Wetland,nan,nan,nan,0


This gives you a Primary Sampling Unit (PSU) Breakdown, percentage wise; This is for YOUR data, next cell will do the same for Satellite data


In [ ]:
# Initializing the lists
point = 1
PSU = []
PSU_classes_ground = []
PSU_classes_model = []

# Getting this set up for the DataFrame
while point <= 36:
    PSU.append(point)
    point += 1

# Splitting all labeling up by PSU
x = 0
while x < len(allData):
    point = 0
    sublist_ground = []
    sublist_model = []
    while point < 100:
        sublist_ground.append(allData['crosswalked label (student data)'][x])
        sublist_model.append(allData['Model label (worldcover)'][x])
        x += 1
        point += 1
    PSU_classes_ground.append(sublist_ground)
    PSU_classes_model.append(sublist_model)

grass_arr_ground = []
shrub_arr_ground = []
water_arr_ground = []
bare_arr_ground = []
crop_arr_ground = []
wetland_arr_ground = []
tree_arr_ground = []
built_arr_ground = []

for _ in PSU_classes_ground:
    grassland_count = 0
    shrubland_count = 0
    water_count = 0
    bare_count = 0
    cropland_count = 0
    wetland_count = 0
    tree_count = 0
    built_count = 0

    # Getting the amount of each land cover type per PSU (tedious Ik)

    for c in _:
        if c == "Grassland":
            grassland_count += 1
        elif c == "Shrubland":
            shrubland_count += 1
        elif c == "Permanent water bodies":
            water_count += 1
        elif c == "Bare/sparse vegetation":
            bare_count += 1
        elif c == "Cropland":
            cropland_count += 1
        elif c == "Herbaceous Wetland":
            wetland_count += 1
        elif c == "Tree cover":
            tree_count += 1
        elif c == "Built-up":
            built_count += 1

    grass_arr_ground.append(grassland_count)
    shrub_arr_ground.append(shrubland_count)
    water_arr_ground.append(water_count)
    bare_arr_ground.append(bare_count)
    crop_arr_ground.append(cropland_count)
    wetland_arr_ground.append(wetland_count)
    tree_arr_ground.append(tree_count)
    built_arr_ground.append(built_count)

PSU_numbers_ground = pd.DataFrame({
    'PSU': PSU,
    'Tree cover': tree_arr_ground,
    'Grassland': grass_arr_ground,
    'Shrubland': shrub_arr_ground,
    'Bare/sparse vegetation': bare_arr_ground,
    'Permanent water bodies': water_arr_ground,
    'Built-up': built_arr_ground,
    'Cropland': crop_arr_ground,
    'Herbaceous Wetland': wetland_arr_ground
})

print("Ground-Truth Data Breakdown by PSU")
print("All numbers represent percentages\n")
PSU_numbers_ground.style.hide(axis='index')

Ground-Truth Data Breakdown by PSU
All numbers represent percentages



PSU,Tree cover,Grassland,Shrubland,Bare/sparse vegetation,Permanent water bodies,Built-up,Cropland,Herbaceous Wetland
1,3,0,0,0,0,97,0,0
2,15,2,5,0,1,77,0,0
3,20,14,3,0,0,63,0,0
4,23,13,2,0,0,62,0,0
5,10,8,1,0,0,81,0,0
6,22,3,1,0,0,74,0,0
7,14,44,0,0,0,42,0,0
8,8,3,5,0,0,84,0,0
9,20,2,4,0,0,74,0,0
10,11,9,9,7,0,64,0,0


This will now be the same PSU data breakdown, but for the Satellite imagery, not the ground truth data

In [ ]:
grass_arr_model = []
shrub_arr_model = []
water_arr_model = []
bare_arr_model = []
crop_arr_model = []
wetland_arr_model = []
tree_arr_model = []
built_arr_model = []

for _ in PSU_classes_model:
    grassland_count = 0
    shrubland_count = 0
    water_count = 0
    bare_count = 0
    cropland_count = 0
    wetland_count = 0
    tree_count = 0
    built_count = 0

    # Getting the amount of each land cover type per PSU (tedious Ik)

    for c in _:
        if c == "Grassland":
            grassland_count += 1
        elif c == "Shrubland":
            shrubland_count += 1
        elif c == "Permanent water bodies":
            water_count += 1
        elif c == "Bare/sparse vegetation":
            bare_count += 1
        elif c == "Cropland":
            cropland_count += 1
        elif c == "Herbaceous Wetland":
            wetland_count += 1
        elif c == "Tree cover":
            tree_count += 1
        elif c == "Built-up":
            built_count += 1

    grass_arr_model.append(grassland_count)
    shrub_arr_model.append(shrubland_count)
    water_arr_model.append(water_count)
    bare_arr_model.append(bare_count)
    crop_arr_model.append(cropland_count)
    wetland_arr_model.append(wetland_count)
    tree_arr_model.append(tree_count)
    built_arr_model.append(built_count)

PSU_numbers_model = pd.DataFrame({
    'PSU': PSU,
    'Tree cover' : tree_arr_model,
    'Grassland': grass_arr_model,
    'Shrubland': shrub_arr_model,
    'Bare/sparse vegetation': bare_arr_model,
    'Permanent water bodies': water_arr_model,
    'Built-up': built_arr_model,
    'Cropland': crop_arr_model,
    'Herbaceous Wetland': wetland_arr_model
})

print("Model Data Breakdown by PSU")
print("All numbers represent percentages\n")
PSU_numbers_model.style.hide(axis='index')

Model Data Breakdown by PSU
All numbers represent percentages



PSU,Tree cover,Grassland,Shrubland,Bare/sparse vegetation,Permanent water bodies,Built-up,Cropland,Herbaceous Wetland
1,0,0,0,0,0,100,0,0
2,21,0,0,1,0,78,0,0
3,24,0,0,6,0,70,0,0
4,38,0,0,0,0,62,0,0
5,18,0,0,0,0,82,0,0
6,4,0,0,3,0,93,0,0
7,12,0,0,2,0,86,0,0
8,7,0,0,1,0,92,0,0
9,28,1,0,1,0,70,0,0
10,16,1,0,0,0,83,0,0


This will now find dominant classes for each PSU (primary and secondary); this is for both the ground and model data

In [ ]:
# If a land cover class takes up less than 10% of land area, it is not counted
# as secondary
# Picked to avoid 4% classes from being considered secondary

SECONDARY_THRESHOLD = 10

# Most represented land cover type for ground-truth
primary_ground = PSU_numbers_ground.drop(columns="PSU").idxmax(axis=1)
# Second most represented land cover type for ground-truth
class_columns = PSU_numbers_ground.columns.drop("PSU")

# Find the second-largest class for each PSU
secondary_ground = PSU_numbers_ground[class_columns].apply(
    lambda row: row.nlargest(2).index[-1],
    axis=1
)

# Find the second-largest class count for each PSU
secondary_ground_count = PSU_numbers_ground[class_columns].apply(
    lambda row: row.nlargest(2).iloc[-1],
    axis=1
)

# Convert that count into a percentage of the PSU total
secondary_ground_percent = (
    secondary_ground_count
    / PSU_numbers_ground[class_columns].sum(axis=1)
    * 100
)

# Remove the secondary class when it represents less than 10%
secondary_ground = secondary_ground.where(
    secondary_ground_percent >= SECONDARY_THRESHOLD,
    "N/A"
)

class_hierarchy_ground = pd.DataFrame({
    'PSU' : PSU,
    'Primary' : primary_ground,
    'Secondary' : secondary_ground
})

class_hierarchy_ground.style.hide(axis='index')

PSU,Primary,Secondary
1,Built-up,N/A
2,Built-up,Tree cover
3,Built-up,Tree cover
4,Built-up,Tree cover
5,Built-up,Tree cover
6,Built-up,Tree cover
7,Grassland,Built-up
8,Built-up,N/A
9,Built-up,Tree cover
10,Built-up,Tree cover


In [ ]:
# Most represented land cover type for the model
primary_model = PSU_numbers_model.drop(columns="PSU").idxmax(axis=1)

# Same class columns as before
class_columns = PSU_numbers_model.columns.drop("PSU")

# Find the second-largest class for each PSU
secondary_model = PSU_numbers_model[class_columns].apply(
    lambda row: row.nlargest(2).index[-1],
    axis=1
)

# Find the second-largest class count for each PSU
secondary_model_count = PSU_numbers_model[class_columns].apply(
    lambda row: row.nlargest(2).iloc[-1],
    axis=1
)

# Convert that count into a percentage of the PSU total
secondary_model_percent = (
    secondary_model_count
    / PSU_numbers_model[class_columns].sum(axis=1)
    * 100
)

# Remove the secondary class when it represents less than the threshold
secondary_model = secondary_model.where(
    secondary_model_percent >= SECONDARY_THRESHOLD,
    "N/A"
)

class_hierarchy_model = pd.DataFrame({
    'PSU': PSU,
    'Primary': primary_model,
    'Secondary': secondary_model
})

class_hierarchy_model.style.hide(axis='index')

PSU,Primary,Secondary
1,Built-up,N/A
2,Built-up,Tree cover
3,Built-up,Tree cover
4,Built-up,Tree cover
5,Built-up,Tree cover
6,Built-up,N/A
7,Built-up,Tree cover
8,Built-up,N/A
9,Built-up,Tree cover
10,Built-up,Tree cover


Now for comparison between primary and secondary classifications

In [10]:
# Setting up the arrays for counting all hierarchical classifications
ground_primary_arr = [0] * len(allDataClasses)
ground_secondary_arr = [0] * len(allDataClasses)
model_primary_arr = [0] * len(allDataClasses)
model_secondary_arr = [0] * len(allDataClasses)

# Counting each class as primary or secondary
for label in primary_ground:
    for land_class in allDataClasses:
        if label == land_class:
            ground_primary_arr[allDataClasses.index(land_class)] += 1

for label in secondary_ground:
    for land_class in allDataClasses:
        if label == land_class:
            ground_secondary_arr[allDataClasses.index(land_class)] += 1

for label in primary_model:
    for land_class in allDataClasses:
        if label == land_class:
            model_primary_arr[allDataClasses.index(land_class)] += 1

for label in secondary_model:
    for land_class in allDataClasses:
        if label == land_class:
            model_secondary_arr[allDataClasses.index(land_class)] += 1

hierarchy_df = pd.DataFrame({
    "Land Cover Class": allDataClasses,
    "Map User Primary Count": ground_primary_arr,
    "Map User Secondary Count": ground_secondary_arr,
    "WorldCover Primary Count": model_primary_arr,
    "WorldCover Secondary Count": model_secondary_arr
})

hierarchy_df.style.hide(axis="index")

Land Cover Class,Map User Primary Count,Map User Secondary Count,WorldCover Primary Count,WorldCover Secondary Count
Tree cover,0,24,3,19
Grassland,4,6,2,2
Shrubland,0,0,0,0
Bare/sparse vegetation,0,0,0,0
Permanent water bodies,3,0,2,1
Built-up,29,3,29,3
Cropland,0,0,0,1
Herbaceous Wetland,0,0,0,0


All the outputted files have different information about the land cover breakdown

This will upload all the files to a folder

In [11]:
from pathlib import Path

# Create the folder if it doesn't already exist
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)


classNumbers.to_csv(output_dir / "all_SSUs_Classes.csv", index=False)
cm_df.to_csv(output_dir / "confusion_matrix.csv", index=True, index_label="Ground Truth")
netMetrics.to_csv(output_dir / "confusion_matrix_metrics.csv", index=False)
PSU_numbers_ground.to_csv(output_dir / "ground_PSU_breakdown.csv", index=False)
PSU_numbers_model.to_csv(output_dir / "model_PSU_breakdown.csv", index=False)
class_hierarchy_ground.to_csv(output_dir / "ground_data_class_hierarchy.csv", index=False)
class_hierarchy_model.to_csv(output_dir / "model_data_class_hierarchy.csv", index=False)
hierarchy_df.to_csv(output_dir / "hierarchy_analysis.csv", index=False)